Analysis of macro-main results

In [2]:
# Autoreload modules when changes are applied to them
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# @title Define Paths
INPUT_PATH = "/Users/jmoran/Projects/macrocosm/macromodel/macro-main/tests/test_macro_data/unit/sample_raw_data"  # @param {"type":"string","placeholder":"data"}
PKL_PATH = "data.pkl"  # @param {"type":"string", "placeholder":"data.pkl"}

import macromodel
import macro_data
from macro_data import configuration_utils
import os

print(INPUT_PATH)
print(PKL_PATH)


def create_pickle(configuration, filename):
    creator = macro_data.DataWrapper.from_config(
        configuration=configuration, raw_data_path=INPUT_PATH, single_hfcs_survey=True
    )

    creator.save(filename)

/Users/jmoran/Projects/macrocosm/macromodel/macro-main/tests/test_macro_data/unit/sample_raw_data
data.pkl


In [4]:
from macro_data.configuration.countries import Country as CountryCode
from macro_data.configuration.region import Region
import pickle as pkl
from pathlib import Path
from macro_data import DataWrapper

data_config = configuration_utils.default_data_configuration(
    countries=["CAN"],
    aggregate_industries=False,
    proxy_country_dict={"CAN": "FRA"},
    use_disagg_can_2014_reader=True,
)

data_config.year = 2014

data_config.can_disaggregation = False
data_config.aggregate_industries = False
data_config.prune_date = None
data_config.seed = 0

base_config = data_config.country_configs[CountryCode("CAN")]
base_config.single_firm_per_industry = True
base_config.single_bank = True
base_config.single_government_entity = True

base_config.firms_configuration.constructor = "Default"

base_config.scale = 1000

# Define Canadian provinces
provinces = [
    Region.from_code("CAN_AB", "Alberta"),
    Region.from_code("CAN_BC", "British Columbia"),
    Region.from_code("CAN_MB", "Manitoba"),
    Region.from_code("CAN_NB", "New Brunswick"),
    Region.from_code("CAN_NL", "Newfoundland and Labrador"),
    Region.from_code("CAN_NS", "Nova Scotia"),
    Region.from_code("CAN_ON", "Ontario"),
    Region.from_code("CAN_PE", "Prince Edward Island"),
    Region.from_code("CAN_QC", "Quebec"),
    Region.from_code("CAN_SK", "Saskatchewan"),
]

for province in provinces:
    data_config.country_configs[province] = base_config
    data_config.country_configs[province].eu_proxy_country = CountryCode("FRA")

data_config.aggregation_structure = {CountryCode("CAN"): provinces}

raw_data_path = INPUT_PATH
data_wrapper = DataWrapper.from_config(data_config, raw_data_path, single_hfcs_survey=True)

with open(PKL_PATH, "wb") as f:
    pkl.dump(data_wrapper, f)

/Users/jmoran/Projects/macrocosm/macromodel/macro-main/macro_data/readers/icio_sea_matching.py:193: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  sea_reader.df.loc[country_name].loc[sea_reader.industries, "Labour Compensation"] *= va_factor


In [29]:
# create_pickle(data_configuration, PKL_PATH)

In [6]:
# shared values
TimeSteps = 4
set_seed = 0

In [7]:
### substitution AND tax scenario

from macromodel.configurations import SimulationConfiguration, CountryConfiguration
from macromodel.simulation import Simulation
import os
import numpy as np
import pandas as pd

with open(PKL_PATH, "rb") as f:
    data = pkl.load(f)

SCENARIO = "substitution_tax"
OUTPUT_Directory = "./output/"

countries = provinces
n_industries = data.n_industries
# bundled_industries = ["B05a", "B05b", "B05c", "C19","D"]       # temporary until the provincial io table has sector disagg
bundled_industries = ["B05", "C19", "D"]
industries = data.industries
energy_bundle = [list(industries).index(ind) for ind in bundled_industries]

substitution_bundles = [energy_bundle]

configuration = SimulationConfiguration(
    seed=set_seed,
    country_configurations={
        country: CountryConfiguration.n_industry_default(
            n_industries=n_industries,
            bundles=substitution_bundles,
        )
        for country in countries
    },
    t_max=TimeSteps,
)

for country in countries:
    configuration.country_configurations[country].firms.functions.production.name == "BundledLeontief"
    configuration.country_configurations[country].use_consumer_carbon_reg = True
    configuration.country_configurations[country].carbon_tax_start_delay = 5
    configuration.country_configurations[country].carbon_tax_price_growth_rate = 0.05
    configuration.country_configurations[country].carbon_tax_price_growth_duration = 10
    configuration.country_configurations[country].carbon_tax_start_price = 100
    configuration.country_configurations[country].use_obps_reg = True

In [10]:
simulation.countries

{'CAN_AB': <macromodel.country.country.Country at 0x34f8678c0>,
 'CAN_BC': <macromodel.country.country.Country at 0x34e99fc20>,
 'CAN_NS': <macromodel.country.country.Country at 0x34ef4a090>,
 'CAN_QC': <macromodel.country.country.Country at 0x34ef27a40>,
 'CAN_ON': <macromodel.country.country.Country at 0x34f1655e0>,
 'CAN_PE': <macromodel.country.country.Country at 0x36b7bf080>,
 'CAN_NL': <macromodel.country.country.Country at 0x36b828a70>,
 'CAN_SK': <macromodel.country.country.Country at 0x3571163f0>,
 'CAN_NB': <macromodel.country.country.Country at 0x357167da0>,
 'CAN_MB': <macromodel.country.country.Country at 0x34ed19670>}

In [20]:
sorted(list(country.firms.ts.__dict__["dicts"].keys()))

['capital_inputs_stock',
 'capital_inputs_stock_industry',
 'capital_inputs_stock_value',
 'constrained_capital_inputs_target_production',
 'constrained_intermediate_inputs_target_production',
 'corporate_taxes_paid',
 'debt',
 'debt_installments',
 'demand',
 'deposits',
 'desired_labour_inputs',
 'equity',
 'estimated_demand',
 'estimated_growth_by_firm',
 'expected_capital_inputs_stock_value',
 'expected_profits',
 'gross_fixed_capital_formation',
 'gross_operating_surplus_mixed_income',
 'interest_paid',
 'interest_paid_on_deposits',
 'interest_paid_on_loans',
 'intermediate_inputs_stock',
 'intermediate_inputs_stock_industry',
 'intermediate_inputs_stock_value',
 'inventory',
 'inventory_nominal',
 'labour_costs',
 'labour_inputs',
 'labour_productivity',
 'labour_productivity_factor',
 'limiting_capital_inputs',
 'limiting_intermediate_inputs',
 'long_term_loan_debt',
 'n_firms',
 'n_firms_by_industry',
 'nominal_amount_sold_in_lcu',
 'nominal_amount_sold_in_lcu_to_CAN_AB',
 'nom

In [18]:
simulation = Simulation.from_datawrapper(datawrapper=data, simulation_configuration=configuration)

### create dictionaries with empty lists for each region
num_regions = len(simulation.countries)
keys = simulation.countries.keys()
# time
ts_time = []
# amounts/prod
ts_production_o = {key: [] for key in keys}
ts_nom_production_o = {key: [] for key in keys}
ts_amount_o = {key: [] for key in keys}
# prices
ts_price_o = {key: [] for key in keys}
ts_ave_price_o = {key: [] for key in keys}
# emissions
ts_em_input_CO2_o = {key: [] for key in keys}
ts_em_capital_CO2_o = {key: [] for key in keys}

### start sim
for _ in range(TimeSteps):
    # time
    ts_time.append(str(simulation.timestep))

    for name, country in simulation.countries.items():
        print(name)

        # prices
        price = country.economy.ts.prev("good_prices")
        ts_price_o[name].append(price)
        ts_ave_price_o[name].append(np.nanmean(price))

        # amounts
        amount = country.firms.ts.prev("real_amount_bought_as_intermediate_inputs")
        prod = country.firms.ts.prev("production")
        prod_nom = country.firms.ts.prev("production_nominal")
        ts_amount_o[name].append(amount)
        ts_production_o[name].append(prod)
        ts_nom_production_o[name].append(prod_nom)

        # emissions
        em_in = country.firms.ts.prev("input_emissions")
        em_cap = country.firms.ts.prev("capital_emissions")
        ts_em_input_CO2_o[name].append(em_in)
        ts_em_capital_CO2_o[name].append(em_cap)

    simulation.iterate()


OUTPUT_FileName = SCENARIO + ".h5"
### no substitution AND no tax scenario

CAN_AB


KeyError: 'input_emissions'

In [ ]:
# # import pickle
# # model = Simulation.from_datawrapper(
# #     datawrapper=data, simulation_configuration=configuration
# # )

# # model.run()

# if not OUTPUT_Directory.endswith("/"):
#     OUTPUT_Directory += "/"

# if not os.path.exists(OUTPUT_Directory):
#     os.makedirs(OUTPUT_Directory)

# simulation.save(save_dir=OUTPUT_Directory, file_name=OUTPUT_FileName)

Converting Results into IAMC Format

To visualize the results we are going to do some postprocessing to improve the names to be more understandable and aggregating information.

In [ ]:
# # import os
# # path = '/content/macro-main'
# # os.chdir(path)
# #!pip install iamc_macro_processing

# import iamc_macro_processing.iamc_processing as postprocessing

# # @title Define the Configuration {"display-mode":"form"}

# #SCENARIO = "test"
# DISPLAY_YEAR = 2014
# IAMC_OUTPUT_Directory = "./output/"
# IAMC_OUTPUT_FileName = SCENARIO + ".csv"
# #IAMC_OUTPUT_FileName = "test.csv"
# #TimeSteps = 20

# iamc = postprocessing.process(os.path.join(OUTPUT_Directory, OUTPUT_FileName), SCENARIO, 'macromodel', TimeSteps+1, DISPLAY_YEAR, ["CAN"])

# iamc.to_csv(os.path.join(IAMC_OUTPUT_Directory, IAMC_OUTPUT_FileName), index=False)

# print("Postprocessing Complete")

In [ ]:
# TimeSteps

In [ ]:
# iamc